# YO G News — 03 Model Training

This notebook trains the credibility classifier and saves the trained pipeline for the Flask backend.

**Pipeline**
- Text → TF-IDF features
- TF-IDF → Logistic Regression
- Evaluate on held-out test data
- Save the complete pipeline as `yog_news_model.pkl`


In [ ]:
import pandas as pd
import joblib
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

DATA_PATH = Path("../dataset/news_dataset.csv")
if not DATA_PATH.exists():
    DATA_PATH = Path("../YO_G_News_Starter_Dataset.csv")
if not DATA_PATH.exists():
    DATA_PATH = Path("YO_G_News_Starter_Dataset.csv")

df = pd.read_csv(DATA_PATH)

df["title"] = df["title"].fillna("").astype(str)
df["content"] = df["content"].fillna("").astype(str)
df["text"] = (df["title"] + " " + df["content"]).str.replace(r"\s+", " ", regex=True).str.strip()

df = df[df["text"].str.len() > 0].dropna(subset=["credibility_label"])
df = df.drop_duplicates(subset=["text"]).reset_index(drop=True)

X = df["text"]
y = df["credibility_label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print("Training:", len(X_train))
print("Testing:", len(X_test))
print("Classes:", sorted(y.unique()))


In [ ]:
model = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        stop_words="english",
        ngram_range=(1, 2),
        max_features=10000
    )),
    ("classifier", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

model.fit(X_train, y_train)

predictions = model.predict(X_test)
accuracy = accuracy_score(y_test, predictions)

print(f"Test accuracy: {accuracy * 100:.2f}%")
print("\nClassification report:")
print(classification_report(y_test, predictions, zero_division=0))


In [ ]:
model_dir = Path("../backend/model")
model_dir.mkdir(parents=True, exist_ok=True)

model_path = model_dir / "yog_news_model.pkl"
joblib.dump(model, model_path)

print("Model saved to:", model_path.resolve())


In [ ]:
sample_text = "Government department announced a new online scholarship portal for students."

prediction = model.predict([sample_text])[0]
probabilities = model.predict_proba([sample_text])[0]

print("Sample prediction:", prediction)
print("\nProbabilities:")
for label, probability in zip(model.classes_, probabilities):
    print(f"{label}: {probability * 100:.2f}%")


## Important

This notebook trains the model used by the YO G News Flask API. The model should be treated as a **credibility-assistance prototype**, not as definitive proof that an article is true or false.
